# Assignment 1
## Shane Fleming
### 3/31/2026

---

## Setup

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import spacy
import matplotlib.pyplot as plt

---
## Q1 - HTML Tags

I used the inspector tool in Chrome (right click > Inspect) to look at the source code of the Georgetown University Wikipedia page. Here are three types of HTML tags I found:

**`<p>` (paragraph tag)** - This is used for all the main body text in the article. Each paragraph of content about Georgetown's history, academics, etc. is wrapped in a p tag.

**`<a>` (anchor/link tag)** - These are the hyperlinks throughout the article. Anytime you can click on a word to go to another Wikipedia page (like clicking "Jesuit" or "Washington D.C."), that text is inside an a tag.

**`<h2>` (heading tag)** - These are the section headers that break up the article like "History", "Academics", and "Athletics". They also make up the table of contents at the top of the page.

---
## Q2 - HTTP Request

Here I'm using the requests library to make a GET request to the Wikipedia page and checking the status code of the response.

In [ ]:
url = "https://en.wikipedia.org/wiki/Georgetown_University"
response = requests.get(url)

print(f"Status Code: {response.status_code}")

The status code is **200**, which means the request was successful and the server returned the page content.

**400s** status codes mean there was a problem with the request on the client side - for example 404 means the page wasn't found, 403 means access was forbidden, and 429 means you've sent too many requests.

**500s** status codes mean something went wrong on the server side - not because of anything wrong with the request itself. A 500 is a general server error, 503 means the server is temporarily unavailable.

---
## Q3 - Extract Paragraph Text

Using BeautifulSoup to parse the HTML and pull out all text from paragraph tags. Then joining it all into one string and storing it as a single row in a dataframe.

In [ ]:
# Parse the HTML
soup = BeautifulSoup(response.text, 'lxml')

# Get text from all p tags
paragraphs = [p.text for p in soup.find_all("p")]

# Store as one row in a dataframe
p_df = pd.DataFrame({"text": " ".join(paragraphs)}, index=[0])

pd.set_option("display.max_colwidth", 300)
p_df

---
## Q4 - NER with spaCy

Loading the spaCy English pipeline and applying it to the paragraph text to extract named entities. The results go into a dataframe called ner_df with the entity text and its label.

In [ ]:
# Load spaCy pipeline
nlp = spacy.load("en_core_web_sm")

In [ ]:
# Run NER on the paragraph text
doc = nlp(p_df["text"][0])

# Build the ner dataframe
ner_df = pd.DataFrame({
    "text": [ent.text for ent in doc.ents],
    "label": [ent.label_ for ent in doc.ents]
})

print(f"Total entities found: {len(ner_df)}")
print(ner_df["label"].value_counts())

In [ ]:
ner_df.head(20)

---
## Q5 - Top 10 People and Organizations

Filtering ner_df by label to get the top 10 most mentioned people and organizations.

In [ ]:
# Top 10 people
top_people = (
    ner_df[ner_df["label"] == "PERSON"]["text"]
    .value_counts()
    .head(10)
    .reset_index()
)
top_people.columns = ["Person", "Count"]

print("Top 10 People:")
print(top_people.to_string(index=False))

In [ ]:
# Top 10 organizations
top_orgs = (
    ner_df[ner_df["label"] == "ORG"]["text"]
    .value_counts()
    .head(10)
    .reset_index()
)
top_orgs.columns = ["Organization", "Count"]

print("Top 10 Organizations:")
print(top_orgs.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(top_people["Person"][::-1], top_people["Count"][::-1], color="steelblue")
axes[0].set_title("Top 10 People (Paragraph Tags)")
axes[0].set_xlabel("Mentions")

axes[1].barh(top_orgs["Organization"][::-1], top_orgs["Count"][::-1], color="darkorange")
axes[1].set_title("Top 10 Organizations (Paragraph Tags)")
axes[1].set_xlabel("Mentions")

plt.tight_layout()
plt.show()

---
## Q6 - Repeat with Anchor Tags

Now doing the same thing but extracting text from anchor tags instead of paragraph tags.

In [ ]:
# Get text from all anchor tags
anchors = [a.text.strip() for a in soup.find_all("a") if a.text.strip()]

# Store in dataframe
a_df = pd.DataFrame({"text": " ".join(anchors)}, index=[0])

print(f"Anchor tags found: {len(anchors)}")
a_df

In [ ]:
# Run NER on anchor text
doc_a = nlp(a_df["text"][0])

ner_df_a = pd.DataFrame({
    "text": [ent.text for ent in doc_a.ents],
    "label": [ent.label_ for ent in doc_a.ents]
})

print(f"Total entities found: {len(ner_df_a)}")
print(ner_df_a["label"].value_counts())

In [ ]:
ner_df_a.head(20)

In [ ]:
# Top 10 people from anchor text
top_people_a = (
    ner_df_a[ner_df_a["label"] == "PERSON"]["text"]
    .value_counts()
    .head(10)
    .reset_index()
)
top_people_a.columns = ["Person", "Count"]

print("Top 10 People (Anchor Tags):")
print(top_people_a.to_string(index=False))

In [ ]:
# Top 10 orgs from anchor text
top_orgs_a = (
    ner_df_a[ner_df_a["label"] == "ORG"]["text"]
    .value_counts()
    .head(10)
    .reset_index()
)
top_orgs_a.columns = ["Organization", "Count"]

print("Top 10 Organizations (Anchor Tags):")
print(top_orgs_a.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(top_people_a["Person"][::-1], top_people_a["Count"][::-1], color="steelblue")
axes[0].set_title("Top 10 People (Anchor Tags)")
axes[0].set_xlabel("Mentions")

axes[1].barh(top_orgs_a["Organization"][::-1], top_orgs_a["Count"][::-1], color="darkorange")
axes[1].set_title("Top 10 Organizations (Anchor Tags)")
axes[1].set_xlabel("Mentions")

plt.tight_layout()
plt.show()

Yes, the results differ from Q5. Anchor tags contain much shorter fragments of text - mostly just the link labels like "Georgetown Law" or "Patrick Ewing" - rather than full sentences. Because of this, the entity counts are generally lower and the distribution is flatter since most links only appear once. The paragraph text gives better NER results since spaCy works better with full sentences that give more context. The anchor text also tends to have more misclassifications since there's no surrounding context for the model to use.

---
## Q7 - Plain Language Explanation

**Web Scraping** is basically a way to automatically collect data from websites instead of copying and pasting things manually. You write code that goes to a webpage, downloads the raw HTML behind it, and then pulls out the specific pieces of information you want. In this assignment, we told the program to go grab all the paragraph text from a Wikipedia page. It does in a few seconds what would take a person a long time to do manually, and it can be scaled up to work on thousands of pages at once.

**Named Entity Recognition (NER)** is a technique that reads through text and automatically identifies and categorizes important "things" like people, organizations, and places. For example if you give it the sentence "Bill Clinton attended Georgetown University in Washington D.C.", it would pick out Bill Clinton as a person, Georgetown University as an organization, and Washington D.C. as a location. Instead of reading through a long article yourself to find every person or company mentioned, NER does it for you and organizes the results. It's useful when you have a lot of text and want to quickly understand what or who it's about.